In [1]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent

from backend.ai.evaluation.ground_truth_evaluation import (
    evaluate,
    load_ground_truth,
    missing_routine_ids,
    routine_id_from_filename,
)

from backend.ai.similarity_metrics.shared_logic import load_corpus, extract_all_chunk_embeddings
from backend.ai.similarity_metrics.bm25_similarity import bm25_similarity
from backend.ai.similarity_metrics.tfidf_similarity import tf_idf_similarity
from backend.ai.similarity_metrics.cosine_similarity import cosine_similarity
from backend.ai.similarity_metrics.euclidean_similarity import euclidean_similarity
from backend.ai.chunking.customized_chunking import chunk_by_layout
from backend.ai.chunking.fixed_chunking import chunk_by_fixed_size, DEFAULT_CHUNK_SIZE

BEST_EMBEDDING_MODEL = "NbAiLab/borealis-embed-212m"

/home/lucas/miniconda3/envs/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
factor_range = [0.5, 1, 1.5, 2, 2.5, 3, 4.5, 5] # Factors to multiply default with

chunk_sizes_tried = [DEFAULT_CHUNK_SIZE * factor for factor in factor_range]

In [4]:
def grid_search_chunking(
    chunk_sizes_tried,
    chunking_methods=(chunk_by_layout, chunk_by_fixed_size),
    questions_path=REPO_ROOT / "data/raw/serviceNow/serviceNow_questions.xlsx",
    routines_path=REPO_ROOT / "data/raw/kvaliteket",
    ground_truth_path=REPO_ROOT / "data/raw/ground_truth.csv",
):
    score_dict = {}
    ground_truth = load_ground_truth(ground_truth_path)

    for chunker in chunking_methods:
        previous_signature = None

        for max_chunk_size in chunk_sizes_tried:
            config_name = f"{chunker.__name__}{max_chunk_size}"

            corpus = load_corpus(
                questions_path,
                routines_path,
                chunker=chunker,
                max_chunk_size=int(max_chunk_size),
            )

            signature = (
                tuple(corpus.chunk_counts),
                tuple(corpus.chunk_texts),
            )

            if previous_signature is not None:
                assert signature != previous_signature, (
                    f"{chunker.__name__}: corpus did not change at "
                    f"chunk size {max_chunk_size}"
                )

            previous_signature = signature

            scores = evaluate(cosine_similarity(corpus), ground_truth)
            score_dict[config_name] = scores

            print(f"{config_name}: {scores}")

    return score_dict

score_dict = grid_search_chunking(chunk_sizes_tried)

Loading weights: 100%|██████████| 158/158 [00:00<00:00, 10190.37it/s]


chunk_by_layout40.0: hit_at_1                      0.583333
hit_at_3                      0.833333
hit_at_5                      0.916667
hit_at_10                     0.916667
mrr                           0.718254
gap_auc                       0.583333
mean_top_score_routine        0.622794
mean_top_score_no_routine     0.608074
routine_questions            12.000000
no_routine_questions          8.000000
skipped_missing_routine      10.000000
dtype: float64


Loading weights: 100%|██████████| 158/158 [00:00<00:00, 7736.67it/s]


chunk_by_layout80: hit_at_1                      0.500000
hit_at_3                      0.916667
hit_at_5                      0.916667
hit_at_10                     0.916667
mrr                           0.685185
gap_auc                       0.572917
mean_top_score_routine        0.615276
mean_top_score_no_routine     0.597847
routine_questions            12.000000
no_routine_questions          8.000000
skipped_missing_routine      10.000000
dtype: float64


Loading weights: 100%|██████████| 158/158 [00:00<00:00, 7882.34it/s]


...

chunk_by_fixed_size240: hit_at_1                      0.500000
hit_at_3                      0.833333
hit_at_5                      0.916667
hit_at_10                     0.916667
mrr                           0.663889
gap_auc                       0.562500
mean_top_score_routine        0.590803
mean_top_score_no_routine     0.574227
routine_questions            12.000000
no_routine_questions          8.000000
skipped_missing_routine      10.000000
dtype: float64


Loading weights: 100%|██████████| 158/158 [00:00<00:00, 9344.46it/s]


chunk_by_fixed_size360.0: hit_at_1                      0.416667
hit_at_3                      0.833333
hit_at_5                      0.916667
hit_at_10                     0.916667
mrr                           0.609069
gap_auc                       0.468750
mean_top_score_routine        0.589975
mean_top_score_no_routine     0.578033
routine_questions            12.000000
no_routine_questions          8.000000
skipped_missing_routine      10.000000
dtype: float64


Loading weights: 100%|██████████| 158/158 [00:00<00:00, 9749.89it/s]


chunk_by_fixed_size400: hit_at_1                      0.333333
hit_at_3                      0.833333
hit_at_5                      0.833333
hit_at_10                     0.916667
mrr                           0.559942
gap_auc                       0.500000
mean_top_score_routine        0.585525
mean_top_score_no_routine     0.575053
routine_questions            12.000000
no_routine_questions          8.000000
skipped_missing_routine      10.000000
dtype: float64


We save to a json file since this took very long.

In [2]:
def top_k_gridsearch(
    top_k_tried,
    questions_path=REPO_ROOT / "data/raw/serviceNow/serviceNow_questions.xlsx",
    routines_path=REPO_ROOT / "data/raw/kvaliteket",
    ground_truth_path=REPO_ROOT / "data/raw/ground_truth.csv",
    ):

    score_dict = {}
    ground_truth = load_ground_truth(ground_truth_path)

    corpus = load_corpus(
            questions_path,
            routines_path
        )

    for k in top_k_tried:

        config_name = str(k)

        scores = evaluate(cosine_similarity(corpus, k=k, model_name=BEST_EMBEDDING_MODEL), ground_truth)
        score_dict[config_name] = scores

        print(f"{config_name}: {scores}")

    return score_dict

k_range = list(range(1, 11))
top_k_score_dict = top_k_gridsearch(k_range)

Loading weights: 100%|██████████| 158/158 [00:00<00:00, 2293.57it/s]


1: hit_at_1                      0.416667
hit_at_3                      0.750000
hit_at_5                      0.916667
hit_at_10                     0.916667
mrr                           0.611111
gap_auc                       0.593750
mean_top_score_routine        0.639744
mean_top_score_no_routine     0.628527
routine_questions            12.000000
no_routine_questions          8.000000
skipped_missing_routine      10.000000
dtype: float64


Loading weights: 100%|██████████| 158/158 [00:00<00:00, 7208.43it/s]


2: hit_at_1                      0.500000
hit_at_3                      0.916667
hit_at_5                      0.916667
hit_at_10                     0.916667
mrr                           0.699346
gap_auc                       0.583333
mean_top_score_routine        0.626330
mean_top_score_no_routine     0.608332
routine_questions            12.000000
no_routine_questions          8.000000
skipped_missing_routine      10.000000
dtype: float64


Loading weights: 100%|██████████| 158/158 [00:00<00:00, 8966.19it/s]


...

8: hit_at_1                      0.333333
hit_at_3                      0.916667
hit_at_5                      0.916667
hit_at_10                     0.916667
mrr                           0.603632
gap_auc                       0.541667
mean_top_score_routine        0.572048
mean_top_score_no_routine     0.566374
routine_questions            12.000000
no_routine_questions          8.000000
skipped_missing_routine      10.000000
dtype: float64


Loading weights: 100%|██████████| 158/158 [00:00<00:00, 8783.65it/s]


9: hit_at_1                      0.416667
hit_at_3                      0.916667
hit_at_5                      0.916667
hit_at_10                     0.916667
mrr                           0.646465
gap_auc                       0.572917
mean_top_score_routine        0.567112
mean_top_score_no_routine     0.560590
routine_questions            12.000000
no_routine_questions          8.000000
skipped_missing_routine      10.000000
dtype: float64


Loading weights: 100%|██████████| 158/158 [00:00<00:00, 8136.48it/s]


10: hit_at_1                      0.416667
hit_at_3                      0.916667
hit_at_5                      0.916667
hit_at_10                     0.916667
mrr                           0.646465
gap_auc                       0.572917
mean_top_score_routine        0.563158
mean_top_score_no_routine     0.555613
routine_questions            12.000000
no_routine_questions          8.000000
skipped_missing_routine      10.000000
dtype: float64


### Conclusion
The scores do not seem to be getting much better than our defaults, top-k=3 and chunk_size=80. It should be fine to keep them the way they are, especially if we might eventually want to present "bad" chunks a word length of 80 seems a good size.